In [ ]:
import pandas as pd
import numpy as np

folder = "/content/drive/MyDrive/Mini"

# Load dataset sizes (RAM tiny)
df_wel  = pd.read_csv(f"{folder}/welfake_clean.csv")
df_net  = pd.read_csv(f"{folder}/fakenewsnet_clean.csv")
df_pred = pd.read_csv(f"{folder}/news_clean.csv")

n_wel  = len(df_wel)
n_net  = len(df_net)
n_pred = len(df_pred)

MAX_LEN = 300
EMB_DIM = 300

# ✅ Load memmap safely (zero RAM usage)
X_wel_sup = np.memmap(f"{folder}/welfake_sup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_wel, MAX_LEN, EMB_DIM))

X_net_sup = np.memmap(f"{folder}/fakenewsnet_sup_seq.dat",
                        dtype="float32", mode="r",
                        shape=(n_net, MAX_LEN, EMB_DIM))

X_pred_sup = np.memmap(f"{folder}/fakepred_sup_seq.dat",
                         dtype="float32", mode="r",
                         shape=(n_pred, MAX_LEN, EMB_DIM))

# ✅ y labels are small → load normally
y_wel  = np.load(f"{folder}/welfake_labels.npy")
y_net  = np.load(f"{folder}/fakenewsnet_labels.npy")
y_pred = np.load(f"{folder}/fakepred_labels.npy")

print("✅ All datasets loaded via memmap")


In [ ]:
def make_dataset(X, y, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
from tensorflow.keras import layers, models, regularizers

MAX_LEN = 300         # shape is (N, 300, 300)
EMB_DIM = 300
L2_LAMBDA = 0.01
NUM_CLASSES = 2

def build_cnn_lstm():
    model = models.Sequential([

        # ✅ Input: already sequences → NO Embedding layer needed!
        layers.Input(shape=(MAX_LEN, EMB_DIM)),

        layers.Conv1D(64, 4, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.Conv1D(64, 3, activation='relu',
                      kernel_regularizer=regularizers.l2(L2_LAMBDA)),

        layers.MaxPooling1D(pool_size=2),

        layers.LSTM(50, return_sequences=True,
                    kernel_regularizer=regularizers.l2(L2_LAMBDA)),
        layers.LSTM(30, kernel_regularizer=regularizers.l2(L2_LAMBDA)),

        layers.Dense(NUM_CLASSES,
                     activation='sigmoid' if NUM_CLASSES == 2 else 'softmax')
    ])

    loss_fn = "binary_crossentropy" if NUM_CLASSES == 2 else "sparse_categorical_crossentropy"

    model.compile(
        optimizer='adam',
        loss=loss_fn,
        metrics=['accuracy']
    )
    return model


In [ ]:
def train_model(X, y, name):

    print(f"\n🚀 Training {name}...")

    # Split indices only (no array slicing!)
    from sklearn.model_selection import train_test_split
    idx_train, idx_test = train_test_split(
        np.arange(len(y)), test_size=0.2,
        random_state=42, stratify=y
    )

    X_train = X[idx_train]
    y_train = y[idx_train]
    X_test  = X[idx_test]
    y_test  = y[idx_test]

    ds_train = make_dataset(X_train, y_train)
    ds_test  = make_dataset(X_test, y_test)

    model = build_cnn_lstm()

    history = model.fit(
        ds_train,
        validation_data=ds_test,
        epochs=10
    )

    model.save(f"{folder}/cnn_lstm_{name}.h5")
    print(f"✅ Saved model cnn_lstm_{name}.h5")

    return history


In [ ]:
train_model(X_wel_unsup, y_wel, "welfake_unsup")

In [ ]:
train_model(X_wel_sup,   y_wel, "welfake_sup")

In [ ]:
train_model(X_net_unsup, y_net, "fakenewsnet_unsup")

In [ ]:
train_model(X_net_sup,   y_net, "fakenewsnet_sup")

In [ ]:
train_model(X_pred_unsup, y_pred, "newspred_unsup")

In [ ]:
train_model(X_pred_sup,   y_pred, "newspred_sup")